In [0]:
%python
# dml/06_carga_analytics_performance_fiis.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("🚀 Iniciando consolidação dos TOP 3 FIIs de cada macro-categoria (Tijolo, Papel, FoF) para a tabela Gold...")

# %%
# 1. Monta a query unificada para capturar o TOP 3 de cada grande classe de ativos (Tijolo, Papel, FoF)
# Usamos ROW_NUMBER() com PARTITION BY ticker para garantir que cada fundo apareça exatamente UMA vez na Gold!
qry_consolidacao_gold = f"""
  WITH ranking_tijolo_global AS (
    -- Re-calcula o ranking de Tijolo de forma global para pegar os 3 melhores de tijolo do mercado
    -- Usamos MAX nos metadados para garantir que não haja produto cartesiano por duplicados na dimensão
    SELECT 
      s.ticker,
      MAX(d.nome_fundo) as nome_fundo,
      MAX(s.preco_atual) as preco_atual,
      MAX(s.p_vp) as p_vp,
      MAX(s.dividend_yield_12m) as dividend_yield_12m,
      MAX(s.score_final) as score_final,
      MAX(s.data_referencia) as data_referencia,
      MAX(s.segmento_alvo) as segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_tijolo s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    GROUP BY s.ticker
  ),

  top_tijolo AS (
    -- Seleciona os 3 melhores de Tijolo Geral desempatados
    SELECT * FROM (
      SELECT 
        ticker,
        nome_fundo,
        preco_atual,
        p_vp,
        dividend_yield_12m,
        score_final,
        ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
        data_referencia,
        segmento_alvo
      FROM ranking_tijolo_global
    ) WHERE posicao_ranking <= 3
  ),
  
  top_papel_global AS (
    -- Agrupa por ticker para garantir que não haja duplicações cadastrais na Staging de Papel
    SELECT 
      s.ticker,
      MAX(d.nome_fundo) as nome_fundo,
      MAX(s.preco_atual) as preco_atual,
      MAX(s.p_vp) as p_vp,
      MAX(s.dividend_yield_12m) as dividend_yield_12m,
      MAX(s.score_final) as score_final,
      MAX(s.data_referencia) as data_referencia,
      MAX(s.segmento_alvo) as segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_papel s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    GROUP BY s.ticker
  ),

  top_papel AS (
    -- Seleciona os 3 melhores de Papel Geral desempatados
    SELECT * FROM (
      SELECT 
        ticker,
        nome_fundo,
        preco_atual,
        p_vp,
        dividend_yield_12m,
        score_final,
        ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
        data_referencia,
        segmento_alvo
      FROM top_papel_global
    ) WHERE posicao_ranking <= 3
  ),
  
  top_fof_global AS (
    -- Agrupa por ticker para garantir que não haja duplicações cadastrais na Staging de FoFs
    SELECT 
      s.ticker,
      MAX(d.nome_fundo) as nome_fundo,
      MAX(s.preco_atual) as preco_atual,
      MAX(s.p_vp) as p_vp,
      MAX(s.dividend_yield_12m) as dividend_yield_12m,
      MAX(s.score_final) as score_final,
      MAX(s.data_referencia) as data_referencia,
      MAX(s.segmento_alvo) as segmento_alvo
    FROM {catalogo}.{schema}.stg_scoring_fof s
    INNER JOIN {catalogo}.{schema}.dim_fundo_imobiliario d ON s.ticker = d.ticker
    GROUP BY s.ticker
  ),

  top_fof AS (
    -- Seleciona os 3 melhores de FoFs Geral desempatados
    SELECT * FROM (
      SELECT 
        ticker,
        nome_fundo,
        preco_atual,
        p_vp,
        dividend_yield_12m,
        score_final,
        ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
        data_referencia,
        segmento_alvo
      FROM top_fof_global
    ) WHERE posicao_ranking <= 3
  )
  
  -- Unifica as três classes em uma única tabela Gold de exatamente 9 registros únicos (Top 3 de cada)
  SELECT * FROM top_tijolo
  UNION ALL
  SELECT * FROM top_papel
  UNION ALL
  SELECT * FROM top_fof
"""

df_gold = spark.sql(qry_consolidacao_gold)
df_gold.createOrReplaceTempView("temp_consolidacao_gold")

# %%
# 2. Executa a gravação atômica especificando as colunas exatas de destino
tabela_destino = "analytics_performance_fiis"

qry_insert_dim_fiis = f"""
  INSERT OVERWRITE {catalogo}.{schema}.{tabela_destino} (
    ticker,
    segmento_alvo,
    nome_fundo,
    preco_atual,
    p_vp,
    dividend_yield_12m,
    score_final,
    posicao_ranking,
    data_selecao,
    data_carga
  )
  SELECT 
    ticker,
    segmento_alvo,
    nome_fundo,
    preco_atual,
    p_vp,
    dividend_yield_12m,
    score_final,
    posicao_ranking,
    data_referencia AS data_selecao,
    CURRENT_TIMESTAMP() AS data_carga
  FROM temp_consolidacao_gold
"""

print(f"Gravando a seleção final unificada na tabela Gold: {catalogo}.{schema}.{tabela_destino}...")

# Grava de forma atômica limpando os dados anteriores da tabela Gold
spark.sql(qry_insert_dim_fiis)

print("✅ Tabela Gold 'analytics_performance_fiis' consolidada com SUCESSO com o TOP 9 FIIs desempatados do mercado!")